# Lightweight Face Recognition Robust to Low-Quality Images — Kaggle Report Version

**Project scope**: So sánh lightweight backbone ở **face representation stage**.  
Face detection & alignment được xem là bước tiền xử lý cố định; notebook này tập trung train/evaluate backbone + ArcFace.

**Kaggle version**:
- Chạy mặc định ở `RUN_MODE = "report"`.
- Đọc CASIA-WebFace từ `/kaggle/input/...`.
- Lưu checkpoint/log/output vào `/kaggle/working/...`.
- Không dùng Google Drive / `google.colab`.


---
## Cell 0: Install Dependencies — Kaggle


In [3]:
# --- Cell 0: Install dependencies for Kaggle ---
# Run once. If mxnet import fails later, restart session once.

!pip install -q easydict ptflops opencv-python scikit-learn gdown
!pip install -q mxnet==1.9.1

print("\nIMPORTANT:")
print("- Kaggle: after first install, if mxnet import fails, use Session options -> Restart session.")
print("- Make sure GPU is enabled: Notebook settings -> Accelerator -> GPU.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 54.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2

---
## Cell 1: Global Config + Kaggle Paths + GPU Check


In [4]:
# ============================================================
#  GLOBAL CONFIG — KAGGLE REPORT MODE
# ============================================================
# This Kaggle version is configured for final/report training by default.

import os
import numpy as np
import torch

RUN_MODE = "report"        # fixed for final comparison
BATCH_SIZE = 64            # Kaggle GPU recommended. Reduce to 32 if OOM.
REPORT_EPOCHS = 15         # use 10 if you want a faster run; keep equal for all backbones.
NUM_EPOCH = REPORT_EPOCHS

RUN_TRAINING = True
RUN_EVAL = False           # set True only if lfw.bin/cfp_fp.bin/agedb_30.bin are available
RUN_BENCHMARK = False      # set True after training if you want speed/size benchmark
FORCE_RETRAIN = False      # True: delete output folder and retrain from scratch

# Train all three for Phase 1. You can narrow this list if needed.
# Accepted names: "mbf", "shufflefacenet", "vargfacenet"
TRAIN_BACKBONES = ["mbf", "shufflefacenet", "vargfacenet"]

KAGGLE_WORKING = "/kaggle/working"
KAGGLE_INPUT = "/kaggle/input"
INSIGHTFACE_DIR = os.path.join(KAGGLE_WORKING, "insightface")
WORK_DIR = os.path.join(INSIGHTFACE_DIR, "recognition", "arcface_torch")
RESULTS_ROOT = os.path.join(KAGGLE_WORKING, "lightweight_fr_results", RUN_MODE)

def find_kaggle_dataset_dir(required_files, search_root=KAGGLE_INPUT):
    """Find first Kaggle input folder containing all required files."""
    if not os.path.exists(search_root):
        return None
    for root, dirs, files in os.walk(search_root):
        files = set(files)
        if all(f in files for f in required_files):
            return root
    return None

# CASIA-WebFace RecordIO training dataset.
# Expected files: train.rec, train.idx, property/train.lst optional.
DATASET_DIR = os.environ.get("DATASET_DIR", "")
if not DATASET_DIR or not os.path.exists(os.path.join(DATASET_DIR, "train.rec")):
    DATASET_DIR = find_kaggle_dataset_dir(["train.rec", "train.idx"])

if DATASET_DIR is None:
    raise FileNotFoundError(
        "Cannot find CASIA-WebFace RecordIO dataset in /kaggle/input. "
        "Add the Kaggle dataset that contains train.rec and train.idx."
    )

# Verification bins may be in the same dataset or another Kaggle dataset.
VAL_DATASET_DIR = os.environ.get("VAL_DATASET_DIR", "")
if not VAL_DATASET_DIR or not os.path.exists(os.path.join(VAL_DATASET_DIR, "lfw.bin")):
    VAL_DATASET_DIR = find_kaggle_dataset_dir(["lfw.bin"]) or DATASET_DIR

os.makedirs(RESULTS_ROOT, exist_ok=True)

print("=" * 80)
print("KAGGLE REPORT CONFIG")
print("=" * 80)
print(f"RUN_MODE        = {RUN_MODE}")
print(f"NUM_EPOCH       = {NUM_EPOCH}")
print(f"BATCH_SIZE      = {BATCH_SIZE}")
print(f"RUN_TRAINING    = {RUN_TRAINING}")
print(f"RUN_EVAL        = {RUN_EVAL}")
print(f"RUN_BENCHMARK   = {RUN_BENCHMARK}")
print(f"FORCE_RETRAIN   = {FORCE_RETRAIN}")
print(f"TRAIN_BACKBONES = {TRAIN_BACKBONES}")
print(f"DATASET_DIR     = {DATASET_DIR}")
print(f"VAL_DATASET_DIR = {VAL_DATASET_DIR}")
print(f"WORK_DIR        = {WORK_DIR}")
print(f"RESULTS_ROOT    = {RESULTS_ROOT}")

# --- GPU / torch check ---
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"VRAM: {vram:.1f} GB")
else:
    raise RuntimeError("GPU NOT AVAILABLE! In Kaggle: Settings -> Accelerator -> GPU.")

# --- MXNet compatibility patch ---
if not hasattr(np, "bool"):
    np.bool = bool

try:
    import mxnet
    print(f"mxnet OK: {mxnet.__version__}")
except Exception as e:
    print("WARNING: mxnet not available. RecordIO/.bin loading may fail.")
    print("Error:", repr(e))
    print("Try restarting the Kaggle session after installing dependencies.")


RUN_MODE       = debug
NUM_EPOCH      = 5
BATCH_SIZE     = 64
RUN_TRAINING   = True
RUN_EVAL       = True
RUN_BENCHMARK  = False
FORCE_RETRAIN  = False

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


---
## Cell 2: Clone InsightFace + Set Working Directory

In [5]:
import os

if not os.path.exists(INSIGHTFACE_DIR):
    !git clone --depth 1 https://github.com/deepinsight/insightface.git {INSIGHTFACE_DIR}
else:
    print(f"[OK] InsightFace already exists: {INSIGHTFACE_DIR}")

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")


Cloning into '/content/insightface'...
remote: Enumerating objects: 1758, done.
remote: Counting objects: 100% (1758/1758), done.
remote: Compressing objects: 100% (1460/1460), done.
remote: Total 1758 (delta 259), reused 1284 (delta 211), pack-reused 0 (from 0)
Receiving objects: 100% (1758/1758), 26.07 MiB | 25.77 MiB/s, done.
Resolving deltas: 100% (259/259), done.
Working directory: /content/insightface/recognition/arcface_torch


In [6]:
# ============================================================
# Patch InsightFace compatibility with current packages
# ============================================================

import os

# 1) Patch lr_scheduler.py for newer PyTorch
lr_path = os.path.join(WORK_DIR, "lr_scheduler.py")

with open(lr_path, "r") as f:
    content = f.read()

old = "super().__init__(optimizer, last_epoch=last_epoch, verbose=verbose)"
new = "super().__init__(optimizer, last_epoch=last_epoch)"

if old in content:
    content = content.replace(old, new)
    with open(lr_path, "w") as f:
        f.write(content)
    print("[OK] Patched lr_scheduler.py: removed verbose argument")
else:
    print("[OK] lr_scheduler.py already patched or pattern not found")

# Optional: verify first lines
!sed -n '1,25p' "{lr_path}"


[OK] Patched lr_scheduler.py: removed verbose argument
from torch.optim.lr_scheduler import _LRScheduler
from torch.optim import SGD
import torch
import warnings

class PolynomialLRWarmup(_LRScheduler):
    def __init__(self, optimizer, warmup_iters, total_iters=5, power=1.0, last_epoch=-1, verbose=False):
        super().__init__(optimizer, last_epoch=last_epoch)
        self.total_iters = total_iters
        self.power = power
        self.warmup_iters = warmup_iters


    def get_lr(self):
        if not self._get_lr_called_within_step:
            warnings.warn("To get the last learning rate computed by the scheduler, "
                          "please use `get_last_lr()`.", UserWarning)

        if self.last_epoch == 0 or self.last_epoch > self.total_iters:
            return [group["lr"] for group in self.optimizer.param_groups]

        if self.last_epoch <= self.warmup_iters:
            return [base_lr * self.last_epoch / self.warmup_iters for base_lr in self.base_lrs]
      

---
## Cell 3: Create / Patch Backbone Files

- **MobileFaceNet**: Already in repo (`backbones/mobilefacenet.py`).
- **ShuffleFaceNet**: ShuffleNetV2-style face recognition backbone + GDC head. *(Not a faithful reproduction of any specific paper; inspired by ShuffleNetV2 architecture.)*
- **VarGFaceNet**: Simplified VarGFaceNet-style compact backbone with SE modules + GDC head. *(Simplified version; does not implement the full variable-group convolution from the ICCVW 2019 paper.)*

In [7]:
# =============================================
#  ShuffleFaceNet (ShuffleNetV2-style backbone)
# =============================================
shufflefacenet_code = r'''"""
ShuffleFaceNet: ShuffleNetV2-style face recognition backbone.

NOTE: This is a ShuffleNetV2-style backbone adapted for face recognition,
NOT a faithful reproduction of any specific "ShuffleFaceNet" paper.
Uses channel shuffle blocks + GDC embedding head.
Input: 3x112x112  Output: 512-d embedding.
"""
import torch
import torch.nn as nn


def channel_shuffle(x, groups):
    b, c, h, w = x.size()
    cpg = c // groups
    x = x.view(b, groups, cpg, h, w)
    x = torch.transpose(x, 1, 2).contiguous()
    return x.view(b, -1, h, w)


class InvertedResidual(nn.Module):
    def __init__(self, inp, oup, stride):
        super().__init__()
        assert stride in [1, 2]
        self.stride = stride
        branch_features = oup // 2

        if stride == 2:
            self.branch1 = nn.Sequential(
                nn.Conv2d(inp, inp, 3, stride, 1, groups=inp, bias=False),
                nn.BatchNorm2d(inp),
                nn.Conv2d(inp, branch_features, 1, bias=False),
                nn.BatchNorm2d(branch_features),
                nn.PReLU(branch_features),
            )
            self.branch2 = nn.Sequential(
                nn.Conv2d(inp, branch_features, 1, bias=False),
                nn.BatchNorm2d(branch_features),
                nn.PReLU(branch_features),
                nn.Conv2d(branch_features, branch_features, 3, stride, 1,
                          groups=branch_features, bias=False),
                nn.BatchNorm2d(branch_features),
                nn.Conv2d(branch_features, branch_features, 1, bias=False),
                nn.BatchNorm2d(branch_features),
                nn.PReLU(branch_features),
            )
        else:
            assert inp == branch_features * 2
            self.branch1 = None
            self.branch2 = nn.Sequential(
                nn.Conv2d(branch_features, branch_features, 1, bias=False),
                nn.BatchNorm2d(branch_features),
                nn.PReLU(branch_features),
                nn.Conv2d(branch_features, branch_features, 3, 1, 1,
                          groups=branch_features, bias=False),
                nn.BatchNorm2d(branch_features),
                nn.Conv2d(branch_features, branch_features, 1, bias=False),
                nn.BatchNorm2d(branch_features),
                nn.PReLU(branch_features),
            )

    def forward(self, x):
        if self.stride == 1:
            x1, x2 = x.chunk(2, dim=1)
            out = torch.cat((x1, self.branch2(x2)), dim=1)
        else:
            out = torch.cat((self.branch1(x), self.branch2(x)), dim=1)
        return channel_shuffle(out, 2)


class ShuffleFaceNet(nn.Module):
    """ShuffleNetV2-style backbone for face recognition."""

    def __init__(self, fp16=False, num_features=512, width_mult=1.0, **kwargs):
        super().__init__()
        self.fp16 = fp16
        stage_repeats = [4, 8, 4]
        if width_mult == 1.0:
            stage_out_channels = [24, 116, 232, 464, 1024]
        elif width_mult == 0.5:
            stage_out_channels = [24, 48, 96, 192, 1024]
        elif width_mult == 1.5:
            stage_out_channels = [24, 176, 352, 704, 1024]
        else:
            raise ValueError(f"Unsupported width_mult: {width_mult}")

        inp = 3
        out0 = stage_out_channels[0]
        self.conv1 = nn.Sequential(
            nn.Conv2d(inp, out0, 3, 2, 1, bias=False),
            nn.BatchNorm2d(out0), nn.PReLU(out0),
        )
        inp = out0
        self.stages = nn.ModuleList()
        for i, repeats in enumerate(stage_repeats):
            out = stage_out_channels[i + 1]
            layers = [InvertedResidual(inp, out, 2)]
            for _ in range(repeats - 1):
                layers.append(InvertedResidual(out, out, 1))
            self.stages.append(nn.Sequential(*layers))
            inp = out

        final_ch = stage_out_channels[-1]
        self.conv5 = nn.Sequential(
            nn.Conv2d(inp, final_ch, 1, bias=False),
            nn.BatchNorm2d(final_ch), nn.PReLU(final_ch),
        )
        self.gdc = nn.Sequential(
            nn.Conv2d(final_ch, final_ch, 7, groups=final_ch, bias=False),
            nn.BatchNorm2d(final_ch),
        )
        self.linear = nn.Linear(final_ch, num_features, bias=False)
        self.bn = nn.BatchNorm1d(num_features)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        with torch.amp.autocast('cuda', enabled=self.fp16):
            x = self.conv1(x)
            for stage in self.stages:
                x = stage(x)
            x = self.conv5(x)
        x = self.gdc(x.float() if self.fp16 else x)
        x = x.view(x.size(0), -1)
        x = self.linear(x)
        x = self.bn(x)
        return x


def get_shufflefacenet(fp16=False, num_features=512, **kwargs):
    return ShuffleFaceNet(fp16=fp16, num_features=num_features)
'''

with open('backbones/shufflefacenet.py', 'w') as f:
    f.write(shufflefacenet_code)
print("[OK] backbones/shufflefacenet.py")

[OK] backbones/shufflefacenet.py


In [8]:
# =============================================
#  VarGFaceNet (Simplified VarGFaceNet-style)
# =============================================
vargfacenet_code = r'''"""
Simplified VarGFaceNet-style compact backbone for face recognition.

NOTE: This is a simplified version inspired by VarGFaceNet (ICCVW 2019).
It does NOT implement the full variable-group convolution from the original paper.
Uses depthwise-separable conv blocks with SE modules + GDC embedding head.
Input: 3x112x112  Output: 512-d embedding.
"""
import torch
import torch.nn as nn


class SEModule(nn.Module):
    def __init__(self, ch, reduction=4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(ch, ch // reduction, 1, bias=False)
        self.act = nn.PReLU(ch // reduction)
        self.fc2 = nn.Conv2d(ch // reduction, ch, 1, bias=False)
        self.sig = nn.Sigmoid()

    def forward(self, x):
        s = self.sig(self.fc2(self.act(self.fc1(self.pool(x)))))
        return x * s


class VarGBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, use_se=True):
        super().__init__()
        self.use_residual = (stride == 1 and in_ch == out_ch)
        self.use_se = use_se
        self.layers = nn.Sequential(
            nn.BatchNorm2d(in_ch),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.PReLU(out_ch),
            nn.Conv2d(out_ch, out_ch, 3, stride, 1, groups=out_ch, bias=False),
            nn.BatchNorm2d(out_ch), nn.PReLU(out_ch),
            nn.Conv2d(out_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        if use_se:
            self.se = SEModule(out_ch)
        if not self.use_residual:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        out = self.layers(x)
        if self.use_se:
            out = self.se(out)
        if self.use_residual:
            out = out + x
        else:
            out = out + self.shortcut(x)
        return out


class VarGFaceNet(nn.Module):
    """Simplified VarGFaceNet-style compact backbone."""

    def __init__(self, fp16=False, num_features=512, **kwargs):
        super().__init__()
        self.fp16 = fp16
        channels = [40, 80, 160, 320]
        num_blocks = [3, 7, 4]
        self.head = nn.Sequential(
            nn.Conv2d(3, channels[0], 3, 2, 1, bias=False),
            nn.BatchNorm2d(channels[0]), nn.PReLU(channels[0]),
            nn.Conv2d(channels[0], channels[0], 3, 1, 1, groups=channels[0], bias=False),
            nn.BatchNorm2d(channels[0]), nn.PReLU(channels[0]),
        )
        self.stages = nn.ModuleList()
        in_c = channels[0]
        for i in range(3):
            out_c = channels[i + 1]
            blocks = [VarGBlock(in_c, out_c, stride=2)]
            for _ in range(num_blocks[i] - 1):
                blocks.append(VarGBlock(out_c, out_c, stride=1))
            self.stages.append(nn.Sequential(*blocks))
            in_c = out_c
        self.embed_conv = nn.Sequential(
            nn.Conv2d(channels[-1], 512, 1, bias=False),
            nn.BatchNorm2d(512), nn.PReLU(512),
        )
        self.gdc = nn.Sequential(
            nn.Conv2d(512, 512, 7, groups=512, bias=False),
            nn.BatchNorm2d(512),
        )
        self.linear = nn.Linear(512, num_features, bias=False)
        self.bn = nn.BatchNorm1d(num_features)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        with torch.amp.autocast('cuda', enabled=self.fp16):
            x = self.head(x)
            for stage in self.stages:
                x = stage(x)
        x = self.embed_conv(x.float() if self.fp16 else x)
        x = self.gdc(x)
        x = x.view(x.size(0), -1)
        x = self.linear(x)
        x = self.bn(x)
        return x


def get_vargfacenet(fp16=False, num_features=512, **kwargs):
    return VarGFaceNet(fp16=fp16, num_features=num_features)
'''

with open('backbones/vargfacenet.py', 'w') as f:
    f.write(vargfacenet_code)
print("[OK] backbones/vargfacenet.py")

[OK] backbones/vargfacenet.py


In [9]:
# === Patch backbones/__init__.py ===
init_file = 'backbones/__init__.py'
with open(init_file, 'r') as f:
    content = f.read()

if 'shufflefacenet' not in content:
    patch = '''
    elif name == "shufflefacenet":
        from .shufflefacenet import get_shufflefacenet
        return get_shufflefacenet(fp16=kwargs.get("fp16", False),
                                  num_features=kwargs.get("num_features", 512))

    elif name == "vargfacenet":
        from .vargfacenet import get_vargfacenet
        return get_vargfacenet(fp16=kwargs.get("fp16", False),
                               num_features=kwargs.get("num_features", 512))

    else:
        raise ValueError()'''
    content = content.replace('    else:\n        raise ValueError()', patch)
    with open(init_file, 'w') as f:
        f.write(content)
    print("[OK] Patched backbones/__init__.py")
else:
    print("[OK] Already patched")

# Verify all 3 backbones
from backbones import get_model
x = torch.randn(2, 3, 112, 112).cuda()
for name in ['mbf', 'shufflefacenet', 'vargfacenet']:
    m = get_model(name, fp16=False, num_features=512).cuda().eval()
    with torch.no_grad():
        y = m(x)
    p = sum(pp.numel() for pp in m.parameters()) / 1e6
    print(f"  {name:<20} output={list(y.shape)}  params={p:.2f}M")
    del m
torch.cuda.empty_cache()
print("[OK] All backbones verified")

[OK] Patched backbones/__init__.py


/content/insightface/recognition/arcface_torch/backbones/mobilefacenet.py:135: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self.fp16):


  mbf                  output=[2, 512]  params=2.06M
  shufflefacenet       output=[2, 512]  params=1.84M
  vargfacenet          output=[2, 512]  params=2.03M
[OK] All backbones verified


---
## Cell 4: Create Configs

In [10]:
import os
os.chdir(WORK_DIR)
os.makedirs('configs/lightweight_fr', exist_ok=True)

if RUN_MODE != "report":
    raise ValueError(f"This Kaggle notebook is report-only. Current RUN_MODE={RUN_MODE}")

if not os.path.exists(os.path.join(DATASET_DIR, "train.rec")):
    raise FileNotFoundError(f"train.rec not found in DATASET_DIR={DATASET_DIR}")

# Snapshot used by training/eval cells to detect stale config generation
CONFIG_RUN_MODE = RUN_MODE
CONFIG_BATCH_SIZE = BATCH_SIZE
CONFIG_NUM_EPOCH = NUM_EPOCH
RUN_PREFIX = RUN_MODE

print(f"[INFO] Generating configs for RUN_MODE={RUN_MODE}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={NUM_EPOCH}")
print(f"[INFO] Training dataset: {DATASET_DIR}")
print("[INFO] Report outputs use report_* folders.")

# --- Base config ---
base_config = f'''from easydict import EasyDict as edict

config = edict()

# Backbone
config.network = "mbf"
config.embedding_size = 512

# Loss
config.loss_type = "combined_margin"   # Phase 1: ArcFace only
config.margin_list = (1.0, 0.5, 0.0)  # (m1, m2, m3) ArcFace default
config.interclass_filtering_threshold = 0

# Training
config.resume = False
config.save_all_states = True
config.output = None
config.fp16 = True
config.batch_size = {BATCH_SIZE}
config.gradient_acc = 1
config.optimizer = "sgd"
config.lr = 0.1
config.momentum = 0.9
config.weight_decay = 5e-4
config.num_epoch = {NUM_EPOCH}
config.warmup_epoch = 1
config.sample_rate = 1.0

# Data (CASIA-WebFace RecordIO)
config.rec = "{DATASET_DIR}"
config.num_classes = 10572
config.num_image = 490623
config.val_targets = []
config.dali = False
config.dali_aug = False
config.num_workers = 2

# Logging
config.verbose = 100000000
config.frequent = 50
config.seed = 2048

# WandB (disabled)
config.using_wandb = False
config.wandb_key = ""
config.suffix_run_name = None
config.wandb_entity = ""
config.wandb_project = ""
config.wandb_log_all = True
config.save_artifacts = False
config.wandb_resume = False
'''

with open('configs/lightweight_fr/__init__.py', 'w') as f:
    f.write('')
with open('configs/lightweight_fr/base_lightweight.py', 'w') as f:
    f.write(base_config)
print(f"[OK] base_lightweight.py (epochs={NUM_EPOCH}, batch={BATCH_SIZE})")

# --- Per-backbone configs (report output folders) ---
configs_spec = {
    'mbf_arcface':      ('mbf',            f'{RUN_PREFIX}_casia_mbf_arcface'),
    'shuffle_arcface':  ('shufflefacenet',  f'{RUN_PREFIX}_casia_shuffle_arcface'),
    'vargface_arcface': ('vargfacenet',     f'{RUN_PREFIX}_casia_vargface_arcface'),
}

for fname, (net, outdir) in configs_spec.items():
    code = f'''from easydict import EasyDict as edict
config = edict()
config.network = "{net}"
config.output = "work_dirs/{outdir}"
'''
    with open(f'configs/lightweight_fr/{fname}.py', 'w') as f:
        f.write(code)
    print(f"[OK] configs/lightweight_fr/{fname}.py -> work_dirs/{outdir}")


[INFO] Generating configs for RUN_MODE=debug, BATCH_SIZE=64
[INFO] Debug/Report outputs are separated to avoid mixed checkpoints.
[OK] base_lightweight.py (epochs=5, batch=64)
[OK] configs/lightweight_fr/mbf_arcface.py -> work_dirs/debug_casia_mbf_arcface
[OK] configs/lightweight_fr/shuffle_arcface.py -> work_dirs/debug_casia_shuffle_arcface
[OK] configs/lightweight_fr/vargface_arcface.py -> work_dirs/debug_casia_vargface_arcface


---
## Cell 5: Create `train_lightweight.py`

Single-GPU T4 compatible. Uses `torchrun --standalone --nproc_per_node=1`.
Reuses `get_dataloader`, `CombinedMarginLoss`, `PartialFC_V2` from repo.

In [11]:
train_script = r'''#!/usr/bin/env python3
"""
train_lightweight.py — Single-GPU friendly training for lightweight FR.

IMPORTANT:
Run this script with:
    torchrun --standalone --nproc_per_node=1 train_lightweight.py <config>
Do NOT run:
    python train_lightweight.py <config>

Usage:
    torchrun --standalone --nproc_per_node=1 train_lightweight.py configs/lightweight_fr/mbf_arcface.py
"""
import argparse, importlib.util, logging, os, os.path as osp
# MXNet 1.9.1 compatibility patch for newer NumPy.
# Must be applied before importing dataset.py, because dataset.py imports mxnet.
import numpy as np
if not hasattr(np, "bool"):
    np.bool = bool
import torch
from torch import distributed
from torch.utils.data import DataLoader

from backbones import get_model
from dataset import get_dataloader
from losses import CombinedMarginLoss
from lr_scheduler import PolynomialLRWarmup
from partial_fc_v2 import PartialFC_V2
from utils.utils_callbacks import CallBackLogging, CallBackVerification
from utils.utils_distributed_sampler import setup_seed
from utils.utils_logging import AverageMeter, init_logging


def load_config(config_file):
    """Load base_lightweight.py then overlay specific config."""
    base_path = osp.join(osp.dirname(osp.abspath(__file__)),
                         "configs", "lightweight_fr", "base_lightweight.py")
    spec = importlib.util.spec_from_file_location("base", base_path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    cfg = mod.config

    spec2 = importlib.util.spec_from_file_location("job", osp.abspath(config_file))
    mod2 = importlib.util.module_from_spec(spec2)
    spec2.loader.exec_module(mod2)
    cfg.update(mod2.config)

    if cfg.output is None:
        cfg.output = osp.join("work_dirs", osp.splitext(osp.basename(config_file))[0])
    return cfg


def main(args):
    if "RANK" not in os.environ:
        raise RuntimeError(
        "Please run with: torchrun --standalone --nproc_per_node=1 train_lightweight.py <config>. "
        "Do not run with plain python.")

    # --- Init distributed (single-GPU via torchrun --standalone) ---
    distributed.init_process_group(backend="nccl")
    rank = distributed.get_rank()
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    world_size = distributed.get_world_size()

    cfg = load_config(args.config)
    setup_seed(seed=cfg.seed, cuda_deterministic=False)
    torch.cuda.set_device(local_rank)
    os.makedirs(cfg.output, exist_ok=True)
    init_logging(rank, cfg.output)

    # --- Data ---
    train_loader = get_dataloader(
        cfg.rec, local_rank, cfg.batch_size,
        cfg.dali, cfg.dali_aug, cfg.seed, cfg.num_workers)

    # --- Backbone ---
    backbone = get_model(
        cfg.network, dropout=0.0, fp16=cfg.fp16,
        num_features=cfg.embedding_size).cuda()
    backbone = torch.nn.parallel.DistributedDataParallel(
        backbone, broadcast_buffers=False,
        device_ids=[local_rank], bucket_cap_mb=16,
        find_unused_parameters=True)
    backbone.train()
    # NOTE: _set_static_graph() removed — can cause issues with custom backbones.

    # --- Loss + Head ---
    margin_loss = CombinedMarginLoss(
        64, cfg.margin_list[0], cfg.margin_list[1],
        cfg.margin_list[2], cfg.interclass_filtering_threshold)

    module_partial_fc = PartialFC_V2(
        margin_loss, cfg.embedding_size, cfg.num_classes,
        cfg.sample_rate, False)
    module_partial_fc.train().cuda()

    # --- Optimizer ---
    opt = torch.optim.SGD(
        [{"params": backbone.parameters()},
         {"params": module_partial_fc.parameters()}],
        lr=cfg.lr, momentum=0.9, weight_decay=cfg.weight_decay)

    # --- LR scheduler ---
    total_batch = cfg.batch_size * world_size
    cfg.warmup_step = cfg.num_image // total_batch * cfg.warmup_epoch
    cfg.total_step = cfg.num_image // total_batch * cfg.num_epoch

    lr_scheduler = PolynomialLRWarmup(
        optimizer=opt, warmup_iters=cfg.warmup_step, total_iters=cfg.total_step)

    # --- Resume ---
    start_epoch, global_step = 0, 0
    ckpt_path = os.path.join(cfg.output, f"checkpoint_gpu_{rank}.pt")
    if cfg.resume and os.path.exists(ckpt_path):
        d = torch.load(ckpt_path)
        start_epoch = d["epoch"]
        global_step = d["global_step"]
        backbone.module.load_state_dict(d["state_dict_backbone"])
        module_partial_fc.load_state_dict(d["state_dict_softmax_fc"])
        opt.load_state_dict(d["state_optimizer"])
        lr_scheduler.load_state_dict(d["state_lr_scheduler"])
        del d
        logging.info(f"Resumed from epoch {start_epoch}, step {global_step}")

    for k, v in cfg.items():
        logging.info(f": {k:<25} {v}")

    # --- Callbacks ---
    callback_verification = None
    if len(cfg.val_targets) > 0:
        callback_verification = CallBackVerification(
            val_targets=cfg.val_targets,
            rec_prefix=cfg.rec,
            summary_writer=None,
            wandb_logger=None
        )

    callback_logging = CallBackLogging(
        frequent=cfg.frequent,
        total_step=cfg.total_step,
        batch_size=cfg.batch_size,
        start_step=global_step,
        writer=None
    )

    # --- Meters + AMP scaler ---
    loss_am = AverageMeter()
    amp = torch.cuda.amp.GradScaler(enabled=cfg.fp16)

    # --- Training loop ---
    for epoch in range(start_epoch, cfg.num_epoch):
        if isinstance(train_loader, DataLoader):
            train_loader.sampler.set_epoch(epoch)
        for _, (img, local_labels) in enumerate(train_loader):
            global_step += 1
            local_embeddings = backbone(img)
            loss = module_partial_fc(local_embeddings, local_labels)

            if cfg.fp16:
                amp.scale(loss).backward()
                if global_step % cfg.gradient_acc == 0:
                    amp.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(backbone.parameters(), 5)
                    amp.step(opt)
                    amp.update()
                    opt.zero_grad()
            else:
                loss.backward()
                if global_step % cfg.gradient_acc == 0:
                    torch.nn.utils.clip_grad_norm_(backbone.parameters(), 5)
                    opt.step()
                    opt.zero_grad()
            lr_scheduler.step()

            with torch.no_grad():
                loss_am.update(loss.item(), 1)

                callback_logging(
                    global_step, loss_am, epoch, cfg.fp16,
                    lr_scheduler.get_last_lr()[0], amp
                )

                if (
                    callback_verification is not None
                    and global_step % cfg.verbose == 0
                    and global_step > 0
                ):
                    callback_verification(global_step, backbone)

        # Save checkpoint every epoch
        if cfg.save_all_states:
            torch.save({
                "epoch": epoch + 1,
                "global_step": global_step,
                "state_dict_backbone": backbone.module.state_dict(),
                "state_dict_softmax_fc": module_partial_fc.state_dict(),
                "state_optimizer": opt.state_dict(),
                "state_lr_scheduler": lr_scheduler.state_dict(),
            }, os.path.join(cfg.output, f"checkpoint_gpu_{rank}.pt"))
        if rank == 0:
            torch.save(backbone.module.state_dict(),
                       os.path.join(cfg.output, "model.pt"))
        if cfg.dali:
            train_loader.reset()

    if rank == 0:
        final_path = os.path.join(cfg.output, "model.pt")
        torch.save(backbone.module.state_dict(), final_path)
        logging.info(f"Training complete. Model saved to {final_path}")


if __name__ == "__main__":
    torch.backends.cudnn.benchmark = True
    parser = argparse.ArgumentParser()
    parser.add_argument("config", type=str)
    main(parser.parse_args())
'''

with open('train_lightweight.py', 'w') as f:
    f.write(train_script)
print("[OK] train_lightweight.py")

[OK] train_lightweight.py


In [ ]:
!head -40 "{os.path.join(WORK_DIR, 'train_lightweight.py')}"


#!/usr/bin/env python3
"""
train_lightweight.py — Single-GPU friendly training for lightweight FR.

IMPORTANT:
Run this script with:
    torchrun --standalone --nproc_per_node=1 train_lightweight.py <config>
Do NOT run:
    python train_lightweight.py <config>

Usage:
    torchrun --standalone --nproc_per_node=1 train_lightweight.py configs/lightweight_fr/mbf_arcface.py
"""
import argparse, importlib.util, logging, os, os.path as osp
# MXNet 1.9.1 compatibility patch for newer NumPy.
# Must be applied before importing dataset.py, because dataset.py imports mxnet.
import numpy as np
if not hasattr(np, "bool"):
    np.bool = bool
import torch
from torch import distributed
from torch.utils.data import DataLoader

from backbones import get_model
from dataset import get_dataloader
from losses import CombinedMarginLoss
from lr_scheduler import PolynomialLRWarmup
from partial_fc_v2 import PartialFC_V2
from utils.utils_callbacks import CallBackLogging, CallBackVerification
from utils.utils_dis

---
## Cell 6: Create Degradation Module

**Core degradations** (3 only):
1. `gaussian_blur` — Gaussian blur with variable sigma
2. `low_resolution` — Downsample then upsample back to 112x112
3. `low_illumination` — Gamma correction to darken image

In [12]:
os.makedirs('degradation', exist_ok=True)

with open('degradation/__init__.py', 'w') as f:
    f.write('from .transforms import DegradationTransform, SUPPORTED_DEGRADATIONS\n')

deg_code = r'''"""
Image degradation transforms for evaluation — core set only.

All transforms accept/return numpy array (H, W, 3) uint8.
Severity levels: 1 (mild) to 5 (severe).
"""
import cv2
import numpy as np

# Core degradations for this project
SUPPORTED_DEGRADATIONS = [
    "gaussian_blur",
    "low_resolution",
    "low_illumination",
]

_GAUSSIAN_BLUR_SIGMA = {1: 0.5, 2: 1.0, 3: 2.0, 4: 3.5, 5: 5.0}
_LOW_RES_SIZE = {1: 56, 2: 42, 3: 28, 4: 20, 5: 14}
_LOW_ILLUM_GAMMA = {1: 1.3, 2: 1.6, 3: 2.0, 4: 2.7, 5: 3.5}


def apply_gaussian_blur(image, severity=1):
    sigma = _GAUSSIAN_BLUR_SIGMA.get(severity, 2.0)
    ksize = int(np.ceil(sigma * 3)) * 2 + 1
    return cv2.GaussianBlur(image, (ksize, ksize), sigma)


def apply_low_resolution(image, severity=1):
    h, w = image.shape[:2]
    low = _LOW_RES_SIZE.get(severity, 28)
    small = cv2.resize(image, (low, low), interpolation=cv2.INTER_LINEAR)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)


def apply_low_illumination(image, severity=1):
    gamma = _LOW_ILLUM_GAMMA.get(severity, 2.0)
    table = np.array([((i / 255.0) ** gamma) * 255
                      for i in range(256)]).astype(np.uint8)
    return cv2.LUT(image, table)


_DEGRADATION_FN = {
    "gaussian_blur": apply_gaussian_blur,
    "low_resolution": apply_low_resolution,
    "low_illumination": apply_low_illumination,
}


class DegradationTransform:
    """Configurable, reproducible image degradation."""

    def __init__(self, degradation_type, severity=1, seed=42):
        if degradation_type not in SUPPORTED_DEGRADATIONS:
            raise ValueError(
                f"Unknown: {degradation_type}. Supported: {SUPPORTED_DEGRADATIONS}")
        if not 1 <= severity <= 5:
            raise ValueError(f"Severity must be 1-5, got {severity}")
        self.degradation_type = degradation_type
        self.severity = severity
        self._fn = _DEGRADATION_FN[degradation_type]

    def apply(self, image):
        return self._fn(image, severity=self.severity)

    def __repr__(self):
        return f"DegradationTransform({self.degradation_type}, s={self.severity})"
'''

with open('degradation/transforms.py', 'w') as f:
    f.write(deg_code)
print("[OK] degradation/ module (3 core degradations)")

[OK] degradation/ module (3 core degradations)


---
## Cell 7: Create `eval_degraded.py`

In [13]:
eval_code = r'''#!/usr/bin/env python3
"""
eval_degraded.py — Clean + degraded evaluation for face recognition.

Usage:
    python eval_degraded.py --network mbf --weight model.pt --rec /path/to/data
    python eval_degraded.py --network mbf --weight model.pt --rec /path/to/data \
        --degradations gaussian_blur,low_resolution,low_illumination --severities 1,3,5
"""
import argparse, os, pickle, sys
import numpy as np
# MXNet 1.9.1 compatibility patch for newer NumPy.
if not hasattr(np, "bool"):
    np.bool = bool
import cv2
import sklearn.preprocessing
import torch, torch.nn as nn

try:
    import mxnet as mx
except ImportError:
    mx = None

from backbones import get_model
from eval.verification import evaluate
from degradation.transforms import DegradationTransform, SUPPORTED_DEGRADATIONS


@torch.no_grad()
def load_bin_as_numpy(path, image_size=(112, 112)):
    assert mx is not None, "mxnet is required to load .bin verification files"
    try:
        with open(path, 'rb') as f: bins, issame_list = pickle.load(f)
    except UnicodeDecodeError:
        with open(path, 'rb') as f: bins, issame_list = pickle.load(f, encoding='bytes')
    n = len(issame_list) * 2
    images = np.empty((n, image_size[0], image_size[1], 3), dtype=np.uint8)
    for i in range(n):
        img = mx.image.imdecode(bins[i]).asnumpy()
        if img.shape[0] != image_size[0] or img.shape[1] != image_size[1]:
            img = cv2.resize(img, (image_size[1], image_size[0]))
        images[i] = img
    return images, issame_list


@torch.no_grad()
def extract_embeddings(images, backbone, batch_size=64, device='cuda',
                       embedding_size=512):
    """Extract embeddings with flip augmentation (same as verification.test)."""
    n = images.shape[0]
    emb_list = []
    for flip in [False, True]:
        emb = np.zeros((n, embedding_size), dtype=np.float32)
        i = 0
        while i < n:
            j = min(i + batch_size, n)
            batch = images[i:j].copy()
            if flip:
                batch = batch[:, :, ::-1, :].copy()
            t = torch.from_numpy(batch.transpose(0, 3, 1, 2).astype(np.float32))
            t = ((t / 255.0) - 0.5) / 0.5
            emb[i:j] = backbone(t.to(device)).cpu().numpy()
            i = j
        emb_list.append(emb)
    return sklearn.preprocessing.normalize(emb_list[0] + emb_list[1])


def eval_condition(images, issame, backbone, degradation=None,
                   batch_size=64, device='cuda', embedding_size=512):
    imgs = images.copy()
    if degradation is not None:
        for i in range(len(imgs)):
            imgs[i] = degradation.apply(imgs[i])
    emb = extract_embeddings(imgs, backbone, batch_size, device, embedding_size)
    _, _, accuracy, val, val_std, far = evaluate(emb, issame, nrof_folds=10)
    return np.mean(accuracy), np.std(accuracy)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--network', type=str, default='mbf')
    parser.add_argument('--weight', type=str, required=True)
    parser.add_argument('--rec', type=str, required=True)
    parser.add_argument('--targets', type=str, default='lfw,cfp_fp,agedb_30')
    parser.add_argument('--degradations', type=str, default='')
    parser.add_argument('--severities', type=str, default='1,3,5')
    parser.add_argument('--batch_size', type=int, default=64)
    parser.add_argument('--seed', type=int, default=42)
    parser.add_argument('--embedding_size', type=int, default=512)
    args = parser.parse_args()

    targets = [t.strip() for t in args.targets.split(',')]
    severities = [int(s) for s in args.severities.split(',')]
    degradations = ([d.strip() for d in args.degradations.split(',')]
                    if args.degradations else [])
    for d in degradations:
        if d not in SUPPORTED_DEGRADATIONS:
            print(f"ERROR: Unknown degradation '{d}'. Supported: {SUPPORTED_DEGRADATIONS}")
            sys.exit(1)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    backbone = get_model(args.network, dropout=0, fp16=False,
                         num_features=args.embedding_size)
    backbone.load_state_dict(torch.load(args.weight, map_location=device))
    backbone = backbone.to(device).eval()

    results = {}
    for tgt in targets:
        bp = os.path.join(args.rec, tgt + '.bin')
        if not os.path.exists(bp):
            print(f'WARNING: {bp} not found, skipping'); continue
        print(f'\n=== {tgt.upper()} ===')
        images, issame = load_bin_as_numpy(bp)
        results[tgt] = {}

        acc, std = eval_condition(images, issame, backbone,
                                  batch_size=args.batch_size, device=device,
                                  embedding_size=args.embedding_size)
        results[tgt]['clean'] = (acc, std)
        print(f'  Clean: {acc*100:.2f}%')

        for dname in degradations:
            for sev in severities:
                deg = DegradationTransform(dname, severity=sev)
                a, s = eval_condition(images, issame, backbone, deg,
                                      args.batch_size, device, args.embedding_size)
                key = f'{dname}_s{sev}'
                results[tgt][key] = (a, s)
                drop = (acc - a) * 100
                print(f'  {key:<30} {a*100:.2f}% (drop: {drop:+.2f}%)')

    # Summary
    print('\n' + '='*70)
    print('SUMMARY')
    print('='*70)
    for tgt, tr in results.items():
        ca = tr['clean'][0]
        print(f'\n--- {tgt.upper()} ---')
        print(f'  {"Condition":<30} {"Acc":>8} {"Drop":>8}')
        print(f'  {"-"*30} {"-"*8} {"-"*8}')
        for cond, (a, _) in tr.items():
            d = '---' if cond == 'clean' else f'{(ca-a)*100:+.2f}%'
            print(f'  {cond:<30} {a*100:>7.2f}% {d:>8}')

if __name__ == '__main__':
    main()
'''

with open('eval_degraded.py', 'w') as f:
    f.write(eval_code)
print("[OK] eval_degraded.py (embedding_size parameterized)")

[OK] eval_degraded.py (embedding_size parameterized)


In [ ]:
!head -30 "{os.path.join(WORK_DIR, 'eval_degraded.py')}"


#!/usr/bin/env python3
"""
eval_degraded.py — Clean + degraded evaluation for face recognition.

Usage:
    python eval_degraded.py --network mbf --weight model.pt --rec /path/to/data
    python eval_degraded.py --network mbf --weight model.pt --rec /path/to/data \
        --degradations gaussian_blur,low_resolution,low_illumination --severities 1,3,5
"""
import argparse, os, pickle, sys
import numpy as np
# MXNet 1.9.1 compatibility patch for newer NumPy.
if not hasattr(np, "bool"):
    np.bool = bool
import cv2
import sklearn.preprocessing
import torch, torch.nn as nn

try:
    import mxnet as mx
except ImportError:
    mx = None

from backbones import get_model
from eval.verification import evaluate
from degradation.transforms import DegradationTransform, SUPPORTED_DEGRADATIONS


@torch.no_grad()
def load_bin_as_numpy(path, image_size=(112, 112)):


---
## Cell 8: Download + Verify CASIA-WebFace Dataset

In [14]:
import os

print("=" * 70)
print("KAGGLE DATASET VERIFICATION")
print("=" * 70)
print(f"Training DATASET_DIR: {DATASET_DIR}")
print(f"Validation VAL_DATASET_DIR: {VAL_DATASET_DIR}")

TRAIN_REQUIRED = ["train.rec", "train.idx"]
TRAIN_OPTIONAL = ["property", "train.lst"]
VAL_FILES = ["lfw.bin", "cfp_fp.bin", "agedb_30.bin"]

print("\n[TRAIN FILES]")
all_ok = True
for fname in TRAIN_REQUIRED:
    fpath = os.path.join(DATASET_DIR, fname)
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / 1024 / 1024
        print(f"  [OK] {fname:<20} {size:.1f} MB")
    else:
        print(f"  [MISSING] {fname}")
        all_ok = False

for fname in TRAIN_OPTIONAL:
    fpath = os.path.join(DATASET_DIR, fname)
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / 1024 / 1024
        print(f"  [OK optional] {fname:<14} {size:.1f} MB")
    else:
        print(f"  [optional missing] {fname}")

if not all_ok:
    raise FileNotFoundError(f"Dataset incomplete in {DATASET_DIR}. Required: {TRAIN_REQUIRED}")

print("\n[VERIFICATION BIN FILES]")
has_any_val = False
for fname in VAL_FILES:
    fpath = os.path.join(VAL_DATASET_DIR, fname)
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / 1024 / 1024
        print(f"  [OK] {fname:<20} {size:.1f} MB")
        has_any_val = True
    else:
        print(f"  [MISSING] {fname}")

if not has_any_val:
    print("\n[WARN] No verification .bin files found. Training is fine, but Cell 11/12 eval will skip targets.")
    print("       To evaluate on Kaggle, add a dataset containing lfw.bin/cfp_fp.bin/agedb_30.bin.")
    print("       Or train on Kaggle and evaluate later on Colab after moving checkpoints.")

print("\n[OK] Training dataset verified.")


If auto-download fails, download manually from:
  https://drive.google.com/file/d/1KxNCrXzln0lal3N4JiYl9cFOIhT78y1l/view
  OR Baidu Pan: https://pan.baidu.com/s/1AfHdPsxJZBD8kBJeIhmq1w
  Then upload .zip to /content/ and run: !unzip -q /content/<file>.zip -d /content/



Downloading...
From (original): https://drive.google.com/uc?id=1KxNCrXzln0lal3N4JiYl9cFOIhT78y1l
From (redirected): https://drive.google.com/uc?id=1KxNCrXzln0lal3N4JiYl9cFOIhT78y1l&confirm=t&uuid=dbff0795-8eb6-43f9-9aff-c52d91f20496
To: /content/faces_webface_112x112.zip
100%|██████████| 2.79G/2.79G [00:58<00:00, 47.6MB/s]



Dataset verification:
  [OK] train.rec            2599.8 MB
  [OK] train.idx            8.3 MB
  [OK] lfw.bin              61.7 MB
  [OK] cfp_fp.bin           72.7 MB
  [OK] agedb_30.bin         73.0 MB

[OK] Dataset verified — all required files present.


---
## Cell 9: Debug Run — Train MobileFaceNet first (sanity check)

> **Sanity check only**: Verify pipeline works (data loading, loss, training, eval callbacks).
>
> Do NOT use Cell 9 results to conclude which backbone is better.
>
> - When `RUN_MODE="debug"`: train MobileFaceNet into `work_dirs/debug_casia_mbf_arcface`.
> - When `RUN_MODE="report"`: Cell 9 is skipped. Use Cell 10 to train all 3 backbones fairly.

In [ ]:
# ============================================================
# Cell 9: Debug sanity check is disabled in Kaggle report version
# ============================================================

print("SKIPPED: this Kaggle notebook is configured for RUN_MODE='report'.")
print("Use Cell 10 to train the selected report backbones.")


=== SANITY CHECK ONLY: MobileFaceNet + ArcFace (5 epochs) ===
This run is only to validate the pipeline, not for backbone ranking.
Output folder: work_dirs/debug_casia_mbf_arcface
Batch size: 64
Expected time: ~40 minutes on T4

========== STDOUT ==========
Grad Scale: 65536   Required: 0 hours
Training: 2026-05-01 15:33:14,217-Speed 514.99 samples/sec   Loss 16.9913   LearningRate 0.003506   Epoch: 4   Global Step: 37250   Fp16 Grad Scale: 65536   Required: 0 hours
Training: 2026-05-01 15:33:19,429-Speed 614.07 samples/sec   Loss 16.9474   LearningRate 0.003343   Epoch: 4   Global Step: 37300   Fp16 Grad Scale: 65536   Required: 0 hours
Training: 2026-05-01 15:33:25,758-Speed 505.61 samples/sec   Loss 17.4751   LearningRate 0.003180   Epoch: 4   Global Step: 37350   Fp16 Grad Scale: 65536   Required: 0 hours
Training: 2026-05-01 15:33:30,964-Speed 614.79 samples/sec   Loss 17.1453   LearningRate 0.003017   Epoch: 4   Global Step: 37400   Fp16 Grad Scale: 65536   Required: 0 hours
Trai

In [ ]:
!nvidia-smi

Fri May  1 15:35:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             31W /   70W |     157MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
## Cell 10: Train All 3 Backbones with ArcFace

> **Fair comparison**: All 3 backbones trained from scratch, same dataset, same loss, same epochs,
> under the same `RUN_MODE` namespace (`debug_*` or `report_*`).

In [ ]:
# ============================================================
# Cell 10: Train Backbones + Realtime Log + Local Kaggle Backup
# ============================================================

import os
import shutil
import subprocess

if not RUN_TRAINING:
    print("SKIPPED (RUN_TRAINING=False)")
else:
    # Check config consistency
    if "CONFIG_RUN_MODE" in globals() and CONFIG_RUN_MODE != RUN_MODE:
        raise RuntimeError(
            f"Config is stale: CONFIG_RUN_MODE={CONFIG_RUN_MODE}, current RUN_MODE={RUN_MODE}. "
            "Please re-run Cell 4."
        )

    if "CONFIG_BATCH_SIZE" in globals() and CONFIG_BATCH_SIZE != BATCH_SIZE:
        raise RuntimeError(
            f"Config is stale: CONFIG_BATCH_SIZE={CONFIG_BATCH_SIZE}, current BATCH_SIZE={BATCH_SIZE}. "
            "Please re-run Cell 4."
        )

    if "CONFIG_NUM_EPOCH" in globals() and CONFIG_NUM_EPOCH != NUM_EPOCH:
        raise RuntimeError(
            f"Config is stale: CONFIG_NUM_EPOCH={CONFIG_NUM_EPOCH}, current NUM_EPOCH={NUM_EPOCH}. "
            "Please re-run Cell 4."
        )

    os.chdir(WORK_DIR)

    BACKUP_ROOT = os.path.join(RESULTS_ROOT, "work_dirs")
    TRAIN_LOG_BACKUP_ROOT = os.path.join(RESULTS_ROOT, "train_logs")

    os.makedirs(BACKUP_ROOT, exist_ok=True)
    os.makedirs(TRAIN_LOG_BACKUP_ROOT, exist_ok=True)
    os.makedirs("train_logs", exist_ok=True)

    print("=" * 80)
    print("  CELL 10: TRAIN BACKBONES + REALTIME LOG + KAGGLE BACKUP")
    print("=" * 80)
    print(f"[INFO] WORK_DIR              : {WORK_DIR}")
    print(f"[INFO] RUN_MODE              : {RUN_MODE}")
    print(f"[INFO] NUM_EPOCH             : {NUM_EPOCH}")
    print(f"[INFO] FORCE_RETRAIN         : {FORCE_RETRAIN}")
    print(f"[INFO] TRAIN_BACKBONES       : {TRAIN_BACKBONES}")
    print(f"[INFO] MODEL BACKUP ROOT     : {BACKUP_ROOT}")
    print(f"[INFO] TRAIN LOG BACKUP ROOT : {TRAIN_LOG_BACKUP_ROOT}")

    def backup_to_output(out_dir):
        """Copy trained model folder to /kaggle/working/lightweight_fr_results."""
        model_path = os.path.join(out_dir, "model.pt")

        if not os.path.exists(model_path):
            print(f"[BACKUP SKIP] No model.pt found in {out_dir}")
            return False

        folder_name = os.path.basename(out_dir)
        dst_dir = os.path.join(BACKUP_ROOT, folder_name)
        tmp_dir = dst_dir + "_tmp"

        if os.path.exists(tmp_dir):
            shutil.rmtree(tmp_dir)

        print(f"[BACKUP] Copying {out_dir} -> {dst_dir}")
        shutil.copytree(out_dir, tmp_dir)

        if os.path.exists(dst_dir):
            shutil.rmtree(dst_dir)

        os.rename(tmp_dir, dst_dir)
        print(f"[BACKUP OK] Saved to Kaggle output folder: {dst_dir}")
        return True

    def backup_train_log(log_path):
        if not os.path.exists(log_path):
            print(f"[LOG BACKUP SKIP] Not found: {log_path}")
            return False

        dst = os.path.join(TRAIN_LOG_BACKUP_ROOT, os.path.basename(log_path))
        shutil.copy2(log_path, dst)
        print(f"[LOG BACKUP OK] {dst}")
        return True

    def restore_from_output(out_dir):
        """
        If local model is missing but /kaggle/working backup exists,
        restore model folder back to local work_dirs.
        """
        folder_name = os.path.basename(out_dir)

        local_model = os.path.join(out_dir, "model.pt")
        backup_dir = os.path.join(BACKUP_ROOT, folder_name)
        backup_model = os.path.join(backup_dir, "model.pt")

        if os.path.exists(local_model):
            return True

        if os.path.exists(backup_model):
            print(f"[RESTORE] Found local Kaggle backup for {folder_name}")
            print(f"[RESTORE] {backup_dir} -> {out_dir}")

            if os.path.exists(out_dir):
                shutil.rmtree(out_dir)

            shutil.copytree(backup_dir, out_dir)
            print(f"[RESTORE OK] Restored local model: {local_model}")
            return True

        return False

    def run_train_realtime(cmd, log_path):
        print(f"[LOG] Training log: {log_path}")
        print("[CMD]", " ".join(cmd))
        print("-" * 80)

        with open(log_path, "w", encoding="utf-8") as f:
            process = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
                universal_newlines=True,
                env={**os.environ, "PYTHONUNBUFFERED": "1"},
            )

            for line in iter(process.stdout.readline, ""):
                print(line, end="")
                f.write(line)
                f.flush()

            process.stdout.close()
            return_code = process.wait()

        print("-" * 80)
        print(f"[RETURN CODE] {return_code}")
        return return_code

    experiments = [
        {
            "name": "MobileFaceNet",
            "tag": "mbf",
            "config": "configs/lightweight_fr/mbf_arcface.py",
            "out_dir": f"work_dirs/{RUN_MODE}_casia_mbf_arcface",
        },
        {
            "name": "ShuffleFaceNet",
            "tag": "shufflefacenet",
            "config": "configs/lightweight_fr/shuffle_arcface.py",
            "out_dir": f"work_dirs/{RUN_MODE}_casia_shuffle_arcface",
        },
        {
            "name": "VarGFaceNet",
            "tag": "vargfacenet",
            "config": "configs/lightweight_fr/vargface_arcface.py",
            "out_dir": f"work_dirs/{RUN_MODE}_casia_vargface_arcface",
        },
    ]

    selected = set(TRAIN_BACKBONES)
    experiments = [e for e in experiments if e["tag"] in selected]
    if len(experiments) == 0:
        raise ValueError(f"No valid backbone selected. TRAIN_BACKBONES={TRAIN_BACKBONES}")

    print("[INFO] Selected experiments:", [e["name"] for e in experiments])

    for exp in experiments:
        name = exp["name"]
        tag = exp["tag"]
        config_path = exp["config"]
        out_dir = exp["out_dir"]
        model_path = os.path.join(out_dir, "model.pt")
        train_log_path = f"train_logs/{RUN_MODE}_train_{tag}.txt"

        print("\n" + "=" * 80)
        print(f"[MODEL]  {name}")
        print(f"[CONFIG] {config_path}")
        print(f"[OUTPUT] {out_dir}")
        print(f"[MODEL PATH] {model_path}")
        print("=" * 80)

        if not os.path.exists(config_path):
            raise FileNotFoundError(f"Config not found: {config_path}")

        # Restore/skip existing model if FORCE_RETRAIN=False
        if not FORCE_RETRAIN:
            restore_from_output(out_dir)

            if os.path.exists(model_path):
                print(f"[SKIP] {name}: found model.pt")
                backup_to_output(out_dir)
                if os.path.exists(train_log_path):
                    backup_train_log(train_log_path)
                continue

        # Remove old local folder if force retrain
        if FORCE_RETRAIN and os.path.exists(out_dir):
            print(f"[FORCE_RETRAIN] Removing old folder: {out_dir}")
            shutil.rmtree(out_dir)

        print(f"[TRAIN] === Training {name} ===")

        cmd = [
            "torchrun",
            "--standalone",
            "--nproc_per_node=1",
            "train_lightweight.py",
            config_path,
        ]

        ret = run_train_realtime(cmd, train_log_path)

        # Backup log even if failed, so you can inspect later
        backup_train_log(train_log_path)

        if ret != 0:
            raise RuntimeError(f"[FAILED] Training failed for {name}. Check log: {train_log_path}")

        if not os.path.exists(model_path):
            raise RuntimeError(
                f"[FAILED] Training finished but model.pt not found for {name}: {model_path}"
            )

        print(f"[TRAIN OK] {name} saved at {model_path}")

        # Backup model immediately after each backbone finishes
        backup_to_output(out_dir)

    print("\n" + "=" * 80)
    print("[DONE] Cell 10 finished. All available trained models/logs have been backed up.")
    print("=" * 80)

    print("\n[CHECK] Model files in Kaggle output:")
    !find "{BACKUP_ROOT}" -name "model.pt" 2>/dev/null

    print("\n[CHECK] Train logs in Kaggle output:")
    !find "{TRAIN_LOG_BACKUP_ROOT}" -type f 2>/dev/null


Mounted at /content/drive
  CELL 10: TRAIN 3 BACKBONES + REALTIME LOG + BACKUP
[INFO] WORK_DIR              : /content/insightface/recognition/arcface_torch
[INFO] RUN_MODE              : debug
[INFO] FORCE_RETRAIN         : False
[INFO] MODEL BACKUP ROOT     : /content/drive/MyDrive/lightweight_fr_results/debug/work_dirs
[INFO] TRAIN LOG BACKUP ROOT : /content/drive/MyDrive/lightweight_fr_results/debug/train_logs

[MODEL]  MobileFaceNet
[CONFIG] configs/lightweight_fr/mbf_arcface.py
[OUTPUT] work_dirs/debug_casia_mbf_arcface
[MODEL PATH] work_dirs/debug_casia_mbf_arcface/model.pt
[RESTORE] Found Drive backup for debug_casia_mbf_arcface
[RESTORE] /content/drive/MyDrive/lightweight_fr_results/debug/work_dirs/debug_casia_mbf_arcface -> work_dirs/debug_casia_mbf_arcface
[RESTORE OK] Restored local model: work_dirs/debug_casia_mbf_arcface/model.pt
[SKIP] MobileFaceNet: found model.pt
[BACKUP] Copying work_dirs/debug_casia_mbf_arcface -> /content/drive/MyDrive/lightweight_fr_results/debug/w

In [ ]:
# Optional: zip current results so you can download them from Kaggle Output.
import os
os.chdir(KAGGLE_WORKING)
zip_path = os.path.join(KAGGLE_WORKING, "lightweight_fr_results_report.zip")
!zip -r -q "{zip_path}" "lightweight_fr_results" "insightface/recognition/arcface_torch/work_dirs" "insightface/recognition/arcface_torch/train_logs" "insightface/recognition/arcface_torch/configs/lightweight_fr" 2>/dev/null
print(f"[OK] Zip saved: {zip_path}")


Mounted at /content/drive


In [ ]:
!find "/kaggle/working" -name "model.pt" 2>/dev/null


/content/drive/MyDrive/lightweight_fr_results/debug/work_dirs/debug_casia_mbf_arcface/model.pt


---
## Cell 11: Clean Evaluation — Compare 3 Backbones

In [ ]:
# ============================================================
# Cell 11: Clean Evaluation + local Kaggle log backup
# ============================================================

import os
import shutil
import subprocess

if RUN_EVAL:
    os.chdir(WORK_DIR)

    DATASET = VAL_DATASET_DIR
    MODEL_BACKUP_ROOT = os.path.join(RESULTS_ROOT, "work_dirs")
    EVAL_BACKUP_ROOT = os.path.join(RESULTS_ROOT, "eval_logs")

    os.makedirs(EVAL_BACKUP_ROOT, exist_ok=True)
    os.makedirs("eval_logs", exist_ok=True)

    print("=" * 70)
    print("  CLEAN EVALUATION: Backbones")
    print("=" * 70)
    print(f"[INFO] Local work dir: {WORK_DIR}")
    print(f"[INFO] Eval dataset: {DATASET}")
    print(f"[INFO] Model backup root: {MODEL_BACKUP_ROOT}")
    print(f"[INFO] Eval log backup root: {EVAL_BACKUP_ROOT}")

    models = [
        {"net": "mbf", "folder": f"{RUN_MODE}_casia_mbf_arcface"},
        {"net": "shufflefacenet", "folder": f"{RUN_MODE}_casia_shuffle_arcface"},
        {"net": "vargfacenet", "folder": f"{RUN_MODE}_casia_vargface_arcface"},
    ]
    models = [m for m in models if m["net"] in set(TRAIN_BACKBONES)]

    def restore_model_if_needed(folder):
        local_dir = os.path.join("work_dirs", folder)
        local_model = os.path.join(local_dir, "model.pt")

        backup_dir = os.path.join(MODEL_BACKUP_ROOT, folder)
        backup_model = os.path.join(backup_dir, "model.pt")

        if os.path.exists(local_model):
            print(f"[LOCAL OK] Found local model: {local_model}")
            return True

        if os.path.exists(backup_model):
            print(f"[RESTORE] Local model missing. Restoring from backup: {backup_dir}")
            if os.path.exists(local_dir):
                shutil.rmtree(local_dir)
            shutil.copytree(backup_dir, local_dir)
            print(f"[RESTORE OK] {backup_dir} -> {local_dir}")
            return True

        print(f"[MISSING] No model found locally or in backup for: {folder}")
        return False

    def run_and_log(cmd, log_path):
        with open(log_path, "w", encoding="utf-8") as f:
            process = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )

            for line in process.stdout:
                print(line, end="")
                f.write(line)

            process.wait()

        return process.returncode

    for item in models:
        net = item["net"]
        folder = item["folder"]

        print(f"\n{'=' * 70}")
        print(f"  {net.upper()} — Clean Evaluation")
        print(f"{'=' * 70}")

        has_model = restore_model_if_needed(folder)

        if not has_model:
            print(f"[SKIP] {net}: model.pt not found")
            continue

        wpath = f"work_dirs/{folder}/model.pt"
        log_path = f"eval_logs/{RUN_MODE}_eval_clean_{net}.txt"

        cmd = [
            "python",
            "eval_degraded.py",
            "--network", net,
            "--weight", wpath,
            "--rec", DATASET,
            "--seed", "42",
        ]

        print("[CMD]", " ".join(cmd))
        ret = run_and_log(cmd, log_path)

        if ret != 0:
            print(f"[FAILED] {net} clean evaluation failed with return code {ret}")
            continue

        print(f"[SAVED LOCAL] {log_path}")

        backup_log_path = os.path.join(EVAL_BACKUP_ROOT, os.path.basename(log_path))
        shutil.copy2(log_path, backup_log_path)
        print(f"[BACKUP OK] {backup_log_path}")

    print("\n[CHECK] Clean evaluation logs:")
    !find "{EVAL_BACKUP_ROOT}" -type f -name "*clean*.txt"

    print("\n[DONE] Cell 11 clean evaluation complete.")

else:
    print("SKIPPED (RUN_EVAL=False)")


---
## Cell 12: Degraded Evaluation — 3 Core Degradations Only

Degradations: `gaussian_blur`, `low_resolution`, `low_illumination`
Severities: 1 (mild), 3 (moderate), 5 (severe)

In [ ]:
# ============================================================
# Cell 12: Degraded Evaluation + local Kaggle log backup
# ============================================================

import os
import shutil
import subprocess

if RUN_EVAL:
    os.chdir(WORK_DIR)

    DATASET = VAL_DATASET_DIR
    DEGRADATIONS = "gaussian_blur,low_resolution,low_illumination"
    SEVERITIES = "1,3,5"

    MODEL_BACKUP_ROOT = os.path.join(RESULTS_ROOT, "work_dirs")
    EVAL_BACKUP_ROOT = os.path.join(RESULTS_ROOT, "eval_logs")

    os.makedirs(EVAL_BACKUP_ROOT, exist_ok=True)
    os.makedirs("eval_logs", exist_ok=True)

    print("=" * 70)
    print("  DEGRADED EVALUATION: Backbones × 3 Degradations × 3 Severities")
    print("=" * 70)
    print(f"[INFO] WORK_DIR          : {WORK_DIR}")
    print(f"[INFO] Eval dataset      : {DATASET}")
    print(f"[INFO] MODEL_BACKUP_ROOT : {MODEL_BACKUP_ROOT}")
    print(f"[INFO] EVAL_BACKUP_ROOT  : {EVAL_BACKUP_ROOT}")

    models = [
        {"net": "mbf", "folder": f"{RUN_MODE}_casia_mbf_arcface"},
        {"net": "shufflefacenet", "folder": f"{RUN_MODE}_casia_shuffle_arcface"},
        {"net": "vargfacenet", "folder": f"{RUN_MODE}_casia_vargface_arcface"},
    ]
    models = [m for m in models if m["net"] in set(TRAIN_BACKBONES)]

    def restore_model_if_needed(folder):
        local_dir = os.path.join("work_dirs", folder)
        local_model = os.path.join(local_dir, "model.pt")

        backup_dir = os.path.join(MODEL_BACKUP_ROOT, folder)
        backup_model = os.path.join(backup_dir, "model.pt")

        if os.path.exists(local_model):
            print(f"[LOCAL OK] Found local model: {local_model}")
            return True

        if os.path.exists(backup_model):
            print(f"[RESTORE] Local model missing. Restoring from backup:")
            print(f"          {backup_dir} -> {local_dir}")

            if os.path.exists(local_dir):
                shutil.rmtree(local_dir)

            shutil.copytree(backup_dir, local_dir)
            print(f"[RESTORE OK] {local_model}")
            return True

        print(f"[MISSING] No model found locally or in backup for: {folder}")
        return False

    def run_and_log(cmd, log_path):
        os.makedirs(os.path.dirname(log_path), exist_ok=True)

        with open(log_path, "w", encoding="utf-8") as f:
            process = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )

            for line in process.stdout:
                print(line, end="")
                f.write(line)

            process.wait()

        return process.returncode

    for item in models:
        net = item["net"]
        folder = item["folder"]

        print(f"\n{'=' * 70}")
        print(f"  {net.upper()} — Degraded Evaluation")
        print(f"{'=' * 70}")

        has_model = restore_model_if_needed(folder)

        if not has_model:
            print(f"[SKIP] {net}: model.pt not found")
            continue

        wpath = f"work_dirs/{folder}/model.pt"
        log_path = f"eval_logs/{RUN_MODE}_eval_degraded_{net}.txt"

        cmd = [
            "python",
            "eval_degraded.py",
            "--network", net,
            "--weight", wpath,
            "--rec", DATASET,
            "--degradations", DEGRADATIONS,
            "--severities", SEVERITIES,
            "--seed", "42",
        ]

        print("[CMD]", " ".join(cmd))
        ret = run_and_log(cmd, log_path)

        if ret != 0:
            print(f"[FAILED] {net} degraded evaluation failed with return code {ret}")
            continue

        print(f"[SAVED LOCAL] {log_path}")

        backup_log_path = os.path.join(EVAL_BACKUP_ROOT, os.path.basename(log_path))
        shutil.copy2(log_path, backup_log_path)
        print(f"[BACKUP OK] {backup_log_path}")

    print("\n[CHECK] Degraded evaluation logs:")
    !find "{EVAL_BACKUP_ROOT}" -type f -name "*degraded*.txt" 2>/dev/null

    print("\n[DONE] Cell 12 degraded evaluation complete.")

else:
    print("SKIPPED (RUN_EVAL=False)")


---
## Cell 13: Benchmark Model Efficiency

In [ ]:
# ============================================================
# Cell 13: Model Efficiency Benchmark + local Kaggle backup
# ============================================================

if RUN_BENCHMARK:
    import os
    import time
    import shutil
    import torch
    from backbones import get_model

    os.chdir(WORK_DIR)

    BENCH_BACKUP_ROOT = os.path.join(RESULTS_ROOT, "benchmark")
    os.makedirs(BENCH_BACKUP_ROOT, exist_ok=True)
    os.makedirs("benchmark_logs", exist_ok=True)

    if not torch.cuda.is_available():
        print("[SKIP] CUDA/GPU is not available. Benchmark should be run on GPU.")
    else:
        torch.backends.cudnn.benchmark = True

        print("=" * 70)
        print("  MODEL EFFICIENCY BENCHMARK")
        print("=" * 70)
        print(f"[INFO] WORK_DIR          : {WORK_DIR}")
        print(f"[INFO] BENCH_BACKUP_ROOT : {BENCH_BACKUP_ROOT}")
        print(f"[INFO] GPU               : {torch.cuda.get_device_name(0)}")

        x1 = torch.randn(1, 3, 112, 112).cuda()
        x16 = torch.randn(16, 3, 112, 112).cuda()

        header = (
            f"  {'Network':<20} {'Params(M)':>10} {'Size(MB)':>10} "
            f"{'GPU b=1(ms)':>14} {'GPU b=16(ms)':>14}"
        )

        print(f"\n{header}")
        print(f"  {'-'*20} {'-'*10} {'-'*10} {'-'*14} {'-'*14}")

        bench_lines = [
            header,
            f"  {'-'*20} {'-'*10} {'-'*10} {'-'*14} {'-'*14}",
        ]

        for net in TRAIN_BACKBONES:
            print(f"\n[BENCH] {net}")

            try:
                m = get_model(net, fp16=False, num_features=512).cuda().eval()
            except TypeError:
                m = get_model(net).cuda().eval()

            params = sum(p.numel() for p in m.parameters()) / 1e6

            # Model size in FP32
            tmp_path = "/tmp/_bench_model.pt"
            torch.save(m.state_dict(), tmp_path)
            size_mb = os.path.getsize(tmp_path) / 1024 / 1024
            os.remove(tmp_path)

            # GPU timing - batch=1
            with torch.no_grad():
                for _ in range(20):
                    _ = m(x1)

                torch.cuda.synchronize()
                t0 = time.perf_counter()

                for _ in range(100):
                    _ = m(x1)

                torch.cuda.synchronize()
                gpu_b1 = (time.perf_counter() - t0) / 100 * 1000

            # GPU timing - batch=16
            with torch.no_grad():
                for _ in range(10):
                    _ = m(x16)

                torch.cuda.synchronize()
                t0 = time.perf_counter()

                for _ in range(30):
                    _ = m(x16)

                torch.cuda.synchronize()
                gpu_b16 = (time.perf_counter() - t0) / 30 * 1000

            row = (
                f"  {net:<20} {params:>10.2f} {size_mb:>10.1f} "
                f"{gpu_b1:>14.2f} {gpu_b16:>14.2f}"
            )

            print(row)
            bench_lines.append(row)

            del m
            torch.cuda.empty_cache()

        bench_path = f"benchmark_logs/{RUN_MODE}_benchmark.txt"

        with open(bench_path, "w", encoding="utf-8") as bf:
            bf.write("\n".join(bench_lines) + "\n")

        print(f"\n[SAVED LOCAL] {bench_path}")

        backup_bench_path = os.path.join(BENCH_BACKUP_ROOT, os.path.basename(bench_path))
        shutil.copy2(bench_path, backup_bench_path)

        print(f"[BACKUP OK] {backup_bench_path}")

        print("\n[CHECK] Benchmark files:")
        !find "{BENCH_BACKUP_ROOT}" -type f 2>/dev/null

        print("\n[DONE] Cell 13 benchmark complete.")

else:
    print("SKIPPED (RUN_BENCHMARK=False)")


---
## Cell 14: Save Results to Google Drive

In [ ]:
# ============================================================
# Cell 14: Final Backup — collect models, logs, configs, scripts and zip
# ============================================================

import os
import shutil

os.chdir(WORK_DIR)

FINAL_BACKUP = os.path.join(RESULTS_ROOT, "final_backup")
os.makedirs(FINAL_BACKUP, exist_ok=True)

print("=" * 70)
print("  FINAL BACKUP TO KAGGLE WORKING")
print("=" * 70)
print(f"[INFO] Final backup folder: {FINAL_BACKUP}")


def copy_if_exists(src, dst):
    if os.path.exists(src):
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
        print(f"[OK] Copied: {src} -> {dst}")
    else:
        print(f"[SKIP] Not found: {src}")


# 1) Backup trained models/checkpoints
copy_if_exists("work_dirs", os.path.join(FINAL_BACKUP, "work_dirs"))

# 2) Backup clean/degraded evaluation logs
copy_if_exists("eval_logs", os.path.join(FINAL_BACKUP, "eval_logs"))

# 3) Backup benchmark logs if exists
copy_if_exists("benchmark_logs", os.path.join(FINAL_BACKUP, "benchmark_logs"))

# 4) Backup configs
copy_if_exists("configs/lightweight_fr", os.path.join(FINAL_BACKUP, "configs_lightweight_fr"))

# 5) Backup important scripts
for file in ["train_lightweight.py", "eval_degraded.py", "benchmark_lightweight.py"]:
    copy_if_exists(file, os.path.join(FINAL_BACKUP, file))

# 6) Backup added/modified backbone files
for file in ["backbones/shufflefacenet.py", "backbones/vargfacenet.py"]:
    copy_if_exists(file, os.path.join(FINAL_BACKUP, file))

# 7) Backup degradation code if exists
copy_if_exists("degradation", os.path.join(FINAL_BACKUP, "degradation"))

print("\n[CHECK] Model files:")
!find "{RESULTS_ROOT}" -name "model.pt" 2>/dev/null

print("\n[CHECK] Evaluation logs:")
!find "{RESULTS_ROOT}" -name "*eval*.txt" 2>/dev/null

print("\n[CHECK] Benchmark files:")
!find "{RESULTS_ROOT}" -name "*benchmark*" 2>/dev/null

# 8) Zip outputs for easy download
os.chdir(KAGGLE_WORKING)
zip_path = os.path.join(KAGGLE_WORKING, "lightweight_fr_results_report.zip")
!zip -r -q "{zip_path}" "lightweight_fr_results" 2>/dev/null
print(f"\n[OK] Zip saved: {zip_path}")
print("[DONE] Final backup complete.")


---
## How to Run This Notebook on Kaggle

### Setup
1. Open Kaggle Notebook.
2. Add CASIA-WebFace dataset containing `train.rec` and `train.idx`.
3. Enable GPU: **Settings → Accelerator → GPU**.
4. Run Cell 0. If `mxnet` import fails, restart the Kaggle session once.
5. Run Cell 1 → Cell 14 in order.

### Default training mode
This notebook is already configured as:

```python
RUN_MODE = "report"
REPORT_EPOCHS = 15
BATCH_SIZE = 64
TRAIN_BACKBONES = ["mbf", "shufflefacenet", "vargfacenet"]
```

If you want faster training, set:

```python
REPORT_EPOCHS = 10
```

but keep the same epoch count for all backbones.

### Outputs
All outputs are saved under:

```text
/kaggle/working/lightweight_fr_results/report/
```

Final zipped file:

```text
/kaggle/working/lightweight_fr_results_report.zip
```

### Evaluation note
If your Kaggle dataset only contains `train.rec/train.idx` and does not contain:

```text
lfw.bin
cfp_fp.bin
agedb_30.bin
```

then leave:

```python
RUN_EVAL = False
```

Train on Kaggle, download `lightweight_fr_results_report.zip`, then evaluate later on Colab or another environment that has the verification `.bin` files.

### Backbone note
- `mbf` is the MobileFaceNet implementation already available in InsightFace.
- `shufflefacenet` and `vargfacenet` in this notebook are lightweight custom/simplified implementations for controlled comparison, not official improved variants.
